In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/keyadobriyal/original-dataset-v2/EV_Adoption_and_Range_Anxiety_Dataset.csv
/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv


In [ ]:
# ============================================================
# 1. IMPORTS & ENVIRONMENT SETUP
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
print("Libraries loaded successfully.")

# ============================================================
# 2. DATA LOADING
# ============================================================
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e9/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e9/test.csv'
SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv'
ORIG_PATH  = '/kaggle/input/datasets/keyadobriyal/original-dataset-v2/EV_Adoption_and_Range_Anxiety_Dataset.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
orig = pd.read_csv(ORIG_PATH)
submission_template = pd.read_csv(SUB_PATH)

TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
orig[TARGET] = orig[TARGET].map({'Yes': 1, 'No': 0})

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Orig shape : {orig.shape}")

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================
train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined.drop(columns=['Number_of_Cars_Owned'], inplace=True, errors='ignore')

# Categorical vs Numeric identification
cat_cols = combined.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['id', TARGET, 'is_train']]
num_cols = [c for c in combined.columns if c not in cat_cols + ['id', 'is_train', TARGET]]

# A. Missingness indicators
for c in list(num_cols):
    if combined[c].isna().any():
        combined[f'{c}_is_missing'] = combined[c].isna().astype('int8')

# B. Digit decomposition on primary continuous columns
DIGIT_ENGINEER_COLS = [c for c in ['Annual_Income_USD', 'Daily_Commute_km', 'Age'] if c in combined.columns]
DIGIT_K = [-2, -1, 0, 1, 2, 3]
digit_features = []
for c in DIGIT_ENGINEER_COLS:
    values = pd.to_numeric(combined[c], errors='coerce').fillna(0)
    for k in DIGIT_K:
        col_name = f'{c}_digit{k}'
        combined[col_name] = (np.floor(values / (10.0 ** k)) % 10).astype('int8')
        digit_features.append(col_name)

# C. Domain-specific ratios & non-linear features
def safe_num(c):
    return pd.to_numeric(combined[c], errors='coerce') if c in combined.columns else None

income = safe_num('Annual_Income_USD')
commute = safe_num('Daily_Commute_km')
env = safe_num('Environmental_Concern_Level')

if income is not None:
    combined['income_log1p'] = np.log1p(income.clip(lower=0))
    combined['income_30k_flag'] = (income == 30000).astype('int8')
    combined['income_high_flag'] = (income >= income.quantile(0.95)).astype('int8')
    combined['income_low_flag'] = (income <= income.quantile(0.05)).astype('int8')
    combined['is_30k_spike'] = (income == 30000).astype('int8')
    combined['is_millionaire_cliff'] = (income >= 170537).astype('int8')
    combined['is_dead_zone'] = ((income >= 38000) & (income <= 42000)).astype('int8')

if commute is not None:
    combined['commute_log1p'] = np.log1p(commute.clip(lower=0))
    combined['commute_long_flag'] = (commute >= commute.quantile(0.90)).astype('int8')

if income is not None and commute is not None:
    combined['income_per_commute'] = income / (commute.abs() + 1.0)
    combined['commute_income_ratio'] = commute / (income.abs() / 10000.0 + 1.0)

if env is not None:
    combined['is_env_hater'] = (env == 1).astype('int8')
    combined['env_high_flag'] = (env >= env.quantile(0.75)).astype('int8')

# D. Smooth numeric keys for target stats/FE
smooth_key_specs = {
    'income_exact_int': ('Annual_Income_USD', 1.0),
    'income100_floor': ('Annual_Income_USD', 100.0),
    'income1000_floor': ('Annual_Income_USD', 1000.0),
    'commute_integer': ('Daily_Commute_km', 1.0),
}
for out, (src, divisor) in smooth_key_specs.items():
    if src in combined.columns:
        vals = pd.to_numeric(combined[src], errors='coerce')
        combined[out] = np.floor(vals / divisor).fillna(-999999).astype('int64')

# E. Original dataset prior statistics
orig_global_mean = orig[TARGET].mean()
org_mean_sources = [c for c in (cat_cols + num_cols) if c in orig.columns]
for c in org_mean_sources:
    stats = orig.groupby(c, observed=False)[TARGET].mean()
    combined[f'{c}_org_mean'] = combined[c].map(stats).fillna(orig_global_mean).astype('float32')

# F. Frequency encoding
frequency_cols = cat_cols + [c for c in smooth_key_specs if c in combined.columns]
for c in frequency_cols:
    freq = combined[c].value_counts(dropna=False, normalize=True)
    combined[f'{c}_fe'] = combined[c].map(freq).fillna(0).astype('float32')

# G. Select candidate columns for out-of-fold target encoding
TE_MAX_CARDINALITY = 5000
te_candidates = [c for c in frequency_cols if combined[c].nunique(dropna=False) <= TE_MAX_CARDINALITY]
for c in digit_features:
    if c in combined.columns:
        te_candidates.append(c)
TARGET_ENCODE_COLS = list(dict.fromkeys(te_candidates))

# H. Split back to train/test and drop constant/duplicate features
train_fe = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_fe = combined[combined['is_train'] == 0].drop(columns=['is_train', TARGET])

to_drop_const = [
    c for c in train_fe.columns 
    if c not in ['id', TARGET] and train_fe[c].nunique(dropna=False) <= 1
]

eval_cols = [c for c in train_fe.columns if c not in ['id', TARGET] and pd.api.types.is_numeric_dtype(train_fe[c])]
to_drop_corr = []
if len(eval_cols) > 1:
    corr = train_fe[eval_cols].corr(method='pearson').abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop_corr = [c for c in upper.columns if (upper[c] >= 0.99999).any()]

DROP = sorted(set(to_drop_const).union(to_drop_corr))
DROP = [c for c in DROP if c not in ['id', TARGET]]

train_fe.drop(columns=DROP, inplace=True, errors='ignore')
test_fe.drop(columns=DROP, inplace=True, errors='ignore')

train = train_fe
test = test_fe

FEATURES = [c for c in test.columns if c != 'id']
TARGET_ENCODE_COLS = [c for c in TARGET_ENCODE_COLS if c in FEATURES and c not in DROP]

print(f"Final Base Features: {len(FEATURES)}")

# Label encoding categorical columns for tree algorithms
for c in cat_cols:
    if c in train.columns:
        train[c] = train[c].astype('category').cat.codes
        test[c] = test[c].astype('category').cat.codes

# ============================================================
# 4. LEAKAGE-SAFE OOF TARGET ENCODING
# ============================================================
def apply_oof_target_encoding(X_tr, y_tr, X_va, X_ts, te_cols, smoothing=10.0):
    """Smooth, leakage-free Target Encoding derived from training partition only."""
    X_tr, X_va, X_ts = X_tr.copy(), X_va.copy(), X_ts.copy()
    global_mean = y_tr.mean()
    
    for c in te_cols:
        stats = pd.DataFrame({'cat': X_tr[c], 'target': y_tr}).groupby('cat')['target'].agg(['count', 'mean'])
        smooth = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
        
        mapping = smooth.to_dict()
        X_tr[f'{c}_te'] = X_tr[c].map(mapping).fillna(global_mean).astype('float32')
        X_va[f'{c}_te'] = X_va[c].map(mapping).fillna(global_mean).astype('float32')
        X_ts[f'{c}_te'] = X_ts[c].map(mapping).fillna(global_mean).astype('float32')
        
    return X_tr, X_va, X_ts

# ============================================================
# 5. UNIFIED CROSS-VALIDATION SETUP
# ============================================================
N_SPLITS = 7
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

X = train[FEATURES]
y = train[TARGET]
X_test_base = test[FEATURES]

lgb_oof = np.zeros(len(train))
lgb_test = np.zeros(len(test))

xgb_oof = np.zeros(len(train))
xgb_test = np.zeros(len(test))

cat_oof = np.zeros(len(train))
cat_test = np.zeros(len(test))

# ============================================================
# 6. MODEL 1: LIGHTGBM TRAINING
# ============================================================
print("\n" + "="*50)
print("1. TRAINING LIGHTGBM (5 FOLDS)")
print("="*50)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()
    X_va, y_va = X.iloc[valid_idx].copy(), y.iloc[valid_idx].copy()
    X_ts = X_test_base.copy()
    
    X_tr, X_va, X_ts = apply_oof_target_encoding(X_tr, y_tr, X_va, X_ts, TARGET_ENCODE_COLS)
    
    clf = lgb.LGBMClassifier(
        n_estimators=25000,
        learning_rate=0.0125,
        max_depth=8,
        num_leaves=32,
        min_child_samples=15,
        subsample=0.8,
        colsample_bytree=0.4,
        reg_alpha=0.1,
        reg_lambda=2.0,
        random_state=42 + fold,
        metric='auc',
        n_jobs=-1,
        verbose=-1
    )
    
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(500, verbose=False), lgb.log_evaluation(0)]
    )
    
    valid_probs = clf.predict_proba(X_va)[:, 1]
    lgb_oof[valid_idx] = valid_probs
    lgb_test += clf.predict_proba(X_ts)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_va, valid_probs)
    print(f"LGBM Fold {fold}: Best Tree #{clf.best_iteration_} | ROC-AUC: {fold_auc:.5f}")

lgb_auc = roc_auc_score(y, lgb_oof)
print(f"--> LIGHTGBM TOTAL OOF ROC-AUC: {lgb_auc:.6f}")

# ============================================================
# 7. MODEL 2: XGBOOST TRAINING
# ============================================================
print("\n" + "="*50)
print("2. TRAINING XGBOOST (5 FOLDS)")
print("="*50)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()
    X_va, y_va = X.iloc[valid_idx].copy(), y.iloc[valid_idx].copy()
    X_ts = X_test_base.copy()
    
    X_tr, X_va, X_ts = apply_oof_target_encoding(X_tr, y_tr, X_va, X_ts, TARGET_ENCODE_COLS)
    
    clf = xgb.XGBClassifier(
        n_estimators=25000,
        learning_rate=0.0125,
        max_depth=8,
        min_child_weight=4.5,
        subsample=0.75,
        colsample_bytree=0.55,
        reg_alpha=0.75,
        reg_lambda=0.6,
        max_bin=512,
        random_state=42 + fold,
        eval_metric='auc',
        early_stopping_rounds=400,
        tree_method='hist',
        n_jobs=-1,
        objective='binary:logistic'
    )
    
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )
    
    valid_probs = clf.predict_proba(X_va)[:, 1]
    xgb_oof[valid_idx] = valid_probs
    xgb_test += clf.predict_proba(X_ts)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_va, valid_probs)
    print(f"XGB Fold {fold}: Best Tree #{clf.best_iteration} | ROC-AUC: {fold_auc:.5f}")

xgb_auc = roc_auc_score(y, xgb_oof)
print(f"--> XGBOOST TOTAL OOF ROC-AUC: {xgb_auc:.6f}")

# ============================================================
# 8. MODEL 3: CATBOOST TRAINING
# ============================================================
print("\n" + "="*50)
print("3. TRAINING CATBOOST (5 FOLDS)")
print("="*50)

cat_features_idx = [c for c in cat_cols if c in FEATURES]

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()
    X_va, y_va = X.iloc[valid_idx].copy(), y.iloc[valid_idx].copy()
    X_ts = X_test_base.copy()
    
    cb_model = cb.CatBoostClassifier(
        iterations=25000,
        learning_rate=0.0125,
        depth=6,
        eval_metric='AUC',
        random_seed=42 + fold,
        cat_features=cat_features_idx,
        early_stopping_rounds=400,
        verbose=0
    )
    
    cb_model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        use_best_model=True
    )
    
    valid_probs = cb_model.predict_proba(X_va)[:, 1]
    cat_oof[valid_idx] = valid_probs
    cat_test += cb_model.predict_proba(X_ts)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_va, valid_probs)
    print(f"CatBoost Fold {fold}: Best Tree #{cb_model.get_best_iteration()} | ROC-AUC: {fold_auc:.5f}")

cat_auc = roc_auc_score(y, cat_oof)
print(f"--> CATBOOST TOTAL OOF ROC-AUC: {cat_auc:.6f}")

# ============================================================
# 9. LEVEL-2 META-LEARNER STACKING
# ============================================================
print("\n" + "="*50)
print("4. LEVEL-2 META-LEARNER STACKING")
print("="*50)

X_meta_train = pd.DataFrame({'lgb': lgb_oof, 'xgb': xgb_oof, 'cat': cat_oof})
X_meta_test = pd.DataFrame({'lgb': lgb_test, 'xgb': xgb_test, 'cat': cat_test})

meta_oof = np.zeros(len(train))
meta_test = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_meta_train, y), 1):
    meta_model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    meta_model.fit(X_meta_train.iloc[train_idx], y.iloc[train_idx])
    
    meta_oof[valid_idx] = meta_model.predict_proba(X_meta_train.iloc[valid_idx])[:, 1]
    meta_test += meta_model.predict_proba(X_meta_test)[:, 1] / N_SPLITS

stacked_auc = roc_auc_score(y, meta_oof)

print("=" * 65)
print("FINAL ENSEMBLE SUMMARY")
print("=" * 65)
print(f"LightGBM OOF ROC-AUC : {lgb_auc:.6f}")
print(f"XGBoost  OOF ROC-AUC : {xgb_auc:.6f}")
print(f"CatBoost OOF ROC-AUC : {cat_auc:.6f}")
print(f"Meta-Stack OOF AUC   : {stacked_auc:.6f}")
print("=" * 65)

# ============================================================
# 10. FINAL SUBMISSION CREATION
# ============================================================
final_submission = submission_template.copy()
final_submission[TARGET] = meta_test
final_submission.to_csv("submission.csv", index=False)

pd.DataFrame({
    "id": train["id"],
    TARGET: y,
    "LGBM_OOF": lgb_oof,
    "XGB_OOF": xgb_oof,
    "CAT_OOF": cat_oof,
    "STACKED_OOF": meta_oof
}).to_csv("oof_EV_PURCHASE_ENSEMBLE.csv", index=False)

print("Final submission successfully generated and saved to 'submission.csv'.")

Libraries loaded successfully.
Train shape: (668665, 15)
Test shape : (286571, 14)
Orig shape : (10000, 15)
Final Base Features: 55

1. TRAINING LIGHTGBM (5 FOLDS)
